In [1]:
from pathlib import Path
from typing import Dict, Any
from adabn.utils import compute_bn_stats, replace_bn_stats

from model_ranking import Pytorch3DUnetModelConfig

from pytorch3dunet.unet3d.model import (
    get_model,  # pyright: ignore[reportUnknownVariableType]
)
from pytorch3dunet.unet3d.utils import (
    load_checkpoint,  # pyright: ignore[reportUnknownVariableType]
)
from pytorch3dunet.datasets.utils import (
    get_train_loaders,  # pyright: ignore[reportUnknownVariableType]
    get_test_loaders,
)

INFO: P [MainThread] 2025-09-15 10:36:04,806 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/predictor.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/predictor.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
model_config: Dict[str, Any] = {
    "model": {
        "name": "UNet2d_as3d",
        "in_channels": 1,
        "out_channels": 1,
        "layer_order": "bcr",
        "f_maps": 32,
        "final_sigmoid": False,
        "feature_return": False,
        "is_segmentation": False,
        "feature_perturbation": None
    },
    "source_checkpoint": "/g/kreshuk/talks/segmentation_ModelSelection/experiments/EPFL/BatchNorm/E_model_NA2/best_checkpoint.pytorch"
}
model_cfg = Pytorch3DUnetModelConfig.model_validate(model_config['model'])

In [3]:
model = get_model(model_cfg.model_dump())
source_checkpoint = model_config['source_checkpoint']

print("Mean teacher training initialized from source model:", source_checkpoint)
if Path(source_checkpoint).suffix == ".pt":
    model_key = "model_state"
else:
    model_key = "model_state_dict"
_ = load_checkpoint(source_checkpoint, model, model_key=model_key)
reinit_teacher = False

Mean teacher training initialized from source model: /g/kreshuk/talks/segmentation_ModelSelection/experiments/EPFL/BatchNorm/E_model_NA2/best_checkpoint.pytorch


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:64: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(checkpoint_path, map_location="cpu")


In [4]:
supervised_loader_config: Dict[str, Any] = {
    "device": "cpu",
    "loaders": {
        "dataset": "StandardHDF5Dataset",
        "batch_size": 32,
        "num_workers": 8,
        "raw_internal_path": "raw",
        "label_internal_path": "labels",
        "global_normalization": True,
        "global_percentiles": None,
        "output_dir": "/g/kreshuk/talks/model_ranking/notebooks/self_training/AdaptiveBatchNorm",
        "test": {
            "file_paths": ["/scratch/talks/data/EPFL/test.h5"],
            "slice_builder": {
                "name": "SliceBuilder",
                "patch_shape": [1, 256, 256],
                "stride_shape": [1, 256, 256],
                "halo_shape": [0, 32, 32]
            },
            "transformer": {
                "raw": [
                    {"name": "Normalize"},
                    {"name": "ToTensor", "expand_dims": True}
                ]
            },
            "roi": None
        }
    }
}

In [5]:
test_loaders = get_test_loaders(supervised_loader_config)

In [6]:
# Get the first dataloader from the generator
test_loader = next(test_loaders)

2025-09-15 10:36:09,694 [MainThread] INFO Dataset - Creating test set loaders...
2025-09-15 10:36:09,696 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/EPFL/test.h5...
2025-09-15 10:36:09,696 [MainThread] INFO HDF5Dataset - Calculating mean and std of the raw data...
2025-09-15 10:36:09,696 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/EPFL/test.h5...
2025-09-15 10:36:09,696 [MainThread] INFO HDF5Dataset - Calculating mean and std of the raw data...
Global mean: 0.5599145293235779, global std: 0.11937469989061356
2025-09-15 10:36:09,946 [MainThread] INFO Dataset - Slice builder config: {'name': 'SliceBuilder', 'patch_shape': [1, 256, 256], 'stride_shape': [1, 256, 256], 'halo_shape': [0, 32, 32]}
2025-09-15 10:36:09,949 [MainThread] INFO HDF5Dataset - Number of patches: 750
2025-09-15 10:36:09,950 [MainThread] INFO Dataset - Number of workers for the dataloader: 8
Global mean: 0.5599145293235779, global std: 0.11937469989061356
2

In [7]:
# Set model to eval mode
_ = model.eval()

# Compute target domain statistics
bn_stats = compute_bn_stats(model, test_loader)

# Apply AdaBN
replace_bn_stats(model, bn_stats)


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument weight in method wrapper_CUDA__cudnn_batch_norm)